# DriveVLM — QLoRA Fine-tuning on DriveLM

Qwen2.5-VL-3B-Instruct + 4-bit NF4 + LoRA rank-16 on DriveLM-nuScenes.

**Runtime**: GPU (T4 minimum). A100 recommended for bf16 + speed.

**Before running**: Accept dataset terms at https://huggingface.co/datasets/OpenDriveLab/DriveLM

In [ ]:
# ── 1. Install deps ──────────────────────────────────────────────────────────
!pip install -q \
    transformers>=4.48 \
    peft>=0.14 \
    bitsandbytes>=0.45 \
    datasets>=3.0 \
    accelerate>=1.0 \
    qwen-vl-utils \
    Pillow

In [ ]:
# ── 2. Mount Google Drive (for saving checkpoints) ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/drivevlm-checkpoints'

In [ ]:
# ── 3. HuggingFace auth ──────────────────────────────────────────────────────
# Add your token in Colab: left sidebar -> Secrets -> add "HF_TOKEN"
# (account: gtm4981, needs read access to OpenDriveLab/DriveLM)
from huggingface_hub import login
from google.colab import userdata

try:
    token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    token = getpass.getpass('HuggingFace token: ')

login(token=token, add_to_git_credential=False)

In [ ]:
# ── 4. Clone repo ────────────────────────────────────────────────────────────
import os
!git clone https://github.com/gouthamk16/drive-vlm /content/drivevlm
os.chdir('/content/drivevlm')

In [ ]:
# ── 5. Config ─────────────────────────────────────────────────────────────────
MAX_SAMPLES  = 5000   # cap training set (full DriveLM is ~300K — too slow for 1 run)
NUM_EPOCHS   = 1
BATCH_SIZE   = 1
GRAD_ACCUM   = 16     # effective batch = 16
LR           = 2e-4
MAX_SEQ      = 2048
MODEL_ID     = 'Qwen/Qwen2.5-VL-3B-Instruct'

In [ ]:
# ── 6. Load model ─────────────────────────────────────────────────────────────
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto'
)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*28*28,
    max_pixels=1280*28*28,
)

In [ ]:
# ── 7. Load dataset ───────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/drivevlm')

from train.dataset import load_drivelm, train_val_split

full_ds = load_drivelm('train')
print(f'Full dataset size: {len(full_ds)}')

# Cap to MAX_SAMPLES for a tractable run
class _Capped:
    def __init__(self, ds, n): self._ds, self._n = ds, min(n, len(ds))
    def __len__(self): return self._n
    def __getitem__(self, i): return self._ds[i]

capped = _Capped(full_ds, MAX_SAMPLES)
train_ds, val_ds = train_val_split(capped)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
# ── 8. Train ──────────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer
from train.finetune import VLMCollator

use_bf16 = torch.cuda.is_bf16_supported()
print(f'bf16: {use_bf16}  (A100/H100 = True, T4 = False)')

args = TrainingArguments(
    output_dir='/content/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    dataloader_num_workers=2,
    report_to='none',
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=VLMCollator(processor, max_seq=MAX_SEQ),
)

trainer.train()

In [ ]:
# ── 9. Save adapter to Google Drive ──────────────────────────────────────────
import shutil

model.save_pretrained('/content/checkpoints/lora-final')
processor.save_pretrained('/content/checkpoints/lora-final')

shutil.copytree('/content/checkpoints/lora-final', SAVE_DIR, dirs_exist_ok=True)
print(f'Adapter saved to {SAVE_DIR}')

In [ ]:
# ── 10. Quick sanity check (inference with adapter) ───────────────────────────
from peft import PeftModel
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from prompt import build_messages
from output import parse, render
import requests
from PIL import Image
from io import BytesIO

# Download a test image
url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/2009_Toyota_Prius_--_NHTSA.jpg/320px-2009_Toyota_Prius_--_NHTSA.jpg'
img = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
img.save('/content/test.jpg')

msgs = build_messages('/content/test.jpg')
text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
img_inputs, _ = process_vision_info(msgs)
enc = processor(text=[text], images=img_inputs, return_tensors='pt').to(model.device)

with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=512, temperature=0.1, do_sample=True)

raw = processor.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
render(parse(raw))